# 🎬 Manim Cloud Render — works on BOTH Google Colab AND Kaggle
This notebook detects which platform it is running on and renders Manim animations **in the cloud** — zero heavy work on your phone.

**How to use:**
1. **Colab:** File → Save a copy in Drive, then edit `MY_SCENE.py` below (cell 3).
2. **Kaggle:** New Notebook → Import → upload this .ipynb, then turn on **Settings → Internet**.
3. Edit the scene code, run all cells, download your MP4 at the end.

In [ ]:
# Cell 1 — Setup (run once). Installs Manim in the cloud.
import os, sys, subprocess, platform

def is_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

COLAB = is_colab()
WHERE = "Google Colab" if COLAB else ("Kaggle" if "KAGGLE" in os.environ else "Unknown")
print("🌍 Running on:", WHERE)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "manim"], check=True)
print("✅ Manim installed:", end=" ")
import manim
print(manim.__version__)

**Cell 2 — ⬇️ Edit your scene here.** Replace the code with your own Manim scene (or keep the example). The cell below writes it to `MY_SCENE.py`.

In [ ]:
# Cell 2 — Your scene code (EDIT THIS)

SCENE_CODE = """
from manim import *

BG = "#1C1C1C"
PRIMARY = "#58C4DD"
SECONDARY = "#83C167"
ACCENT = "#FFFF00"
MONO = "monospace"

class MyScene(Scene):
    def construct(self):
        self.camera.background_color = BG
        title = Text("Hello Cloud!", font_size=48, color=PRIMARY, weight=BOLD, font=MONO)
        self.play(Write(title), run_time=1.5)
        self.wait(1.0)
        circle = Circle(color=ACCENT, stroke_width=6).next_to(title, DOWN, buff=1.0)
        self.play(Create(circle), run_time=1.5)
        self.play(circle.animate.scale(1.8).set_color(SECONDARY), run_time=1.5)
        self.wait(2.0)
        self.play(FadeOut(Group(*self.mobjects)), run_time=0.5)
"""

with open("MY_SCENE.py", "w") as f:
    f.write(SCENE_CODE)
print("✅ Scene saved to MY_SCENE.py")

In [ ]:
# Cell 3 — Render the animation in the cloud
import subprocess, sys, glob, os

# Quality: -ql draft (fast) | -qm medium | -qh full HD
QUALITY = "-qm"

if COLAB:
    os.chdir("/content")
else:  # Kaggle writes into /kaggle/working
    os.chdir("/kaggle/working")

result = subprocess.run(
    [sys.executable, "-m", "manim", QUALITY, "MY_SCENE.py", "MyScene", "--media_dir", "media"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("ERROR:\n", result.stderr[-2000:])
    raise SystemExit("Render failed — check the scene code above.")

videos = glob.glob("media/**/*.mp4", recursive=True)
print("🎬 Videos produced:", videos)
print("MP4_PATH=" + videos[-1])

In [ ]:
# Cell 4 — Download / watch your video

from IPython.display import Video, display
import glob

videos = glob.glob("media/**/*.mp4", recursive=True)
assert videos, "No video found — run Cell 3 first."
mp4 = videos[-1]
print("Video file:", mp4)
display(Video(mp4, width=640))

# Colab: triggers a download to your phone. Kaggle: file is in /kaggle/working
# (left-hand file panel) — tap the ▶ icon next to it to download.
try:
    from google.colab import files
    files.download(mp4)
    print("✅ Download started in Colab.")
except ImportError:
    print("✅ On Kaggle: open the file panel on the left → media → your .mp4 → tap ▶ to download.")